# Exchangeability Analysis Notebook
Run exchangeability analysis (if needed) and inspect the resulting CSV.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path
import pandas as pd

In [ ]:
def _find_repo_root(start: Path) -> Path:
    cur = start.resolve()
    for p in [cur] + list(cur.parents):
        if (p / 'pyproject.toml').exists():
            return p
    raise FileNotFoundError('Could not locate repo root (missing pyproject.toml).')

REPO_ROOT = _find_repo_root(Path.cwd())
BASE_SAVE_DIR = Path(os.environ.get('BASE_SAVE_DIR', '/n/netscratch/kempner_pehlevan_lab/Lab/ilavie/exchangeability_outputs'))
RUN_ID = 'exchangeability'
WIDTHS = [32]  # set to [] to analyze all widths
SHUFFLE_REPEATS = 2000
PROBE_BATCH_SIZE = 1024
PROBE_LOADER_BATCH_SIZE = 1
FORCE_RERUN = False

if WIDTHS:
    suffix = '_w' + '_'.join(str(w) for w in WIDTHS)
    OUTPUT_CSV = BASE_SAVE_DIR / f'exchangeability_metrics{suffix}.csv'
else:
    OUTPUT_CSV = BASE_SAVE_DIR / 'exchangeability_metrics.csv'

print(f'REPO_ROOT={REPO_ROOT}')
print(f'BASE_SAVE_DIR={BASE_SAVE_DIR}')
print(f'OUTPUT_CSV={OUTPUT_CSV}')

In [ ]:
if FORCE_RERUN or not OUTPUT_CSV.exists():
    cmd = [
        sys.executable,
        str(REPO_ROOT / 'scripts' / 'analyze_exchangeability.py'),
        '--base-save-dir', str(BASE_SAVE_DIR),
        '--run-id', RUN_ID,
        '--output-csv', str(OUTPUT_CSV),
        '--shuffle-repeats', str(SHUFFLE_REPEATS),
        '--probe-batch-size', str(PROBE_BATCH_SIZE),
        '--probe-loader-batch-size', str(PROBE_LOADER_BATCH_SIZE),
    ]
    if WIDTHS:
        cmd.extend(['--widths'] + [str(w) for w in WIDTHS])

    print('Running:')
    print(' '.join(cmd))
    subprocess.run(cmd, check=True, cwd=REPO_ROOT)
else:
    print(f'Skipping analysis; CSV already exists: {OUTPUT_CSV}')

In [ ]:
df = pd.read_csv(OUTPUT_CSV)
print(f'rows={len(df)}, widths={sorted(df["width"].unique().tolist())}')
df.head()

In [ ]:
coverage = (
    df.groupby(['width', 'representation', 'analysis_type'], as_index=False)
      .agg(num_points=('images_seen', 'nunique'), min_images_seen=('images_seen', 'min'), max_images_seen=('images_seen', 'max'))
      .sort_values(['width', 'representation', 'analysis_type'])
)
coverage

In [ ]:
summary = (
    df.groupby(['representation', 'analysis_type', 'width', 'images_seen'], as_index=False)
      .agg(
          ks_distance_mean=('ks_distance', 'mean'),
          w1_distance_mean=('w1_distance', 'mean'),
          train_loss=('train_loss', 'mean'),
          val_loss=('val_loss', 'mean'),
          train_error=('train_error', 'mean'),
          val_error=('val_error', 'mean'),
      )
      .sort_values(['width', 'images_seen', 'representation', 'analysis_type'])
)
summary_path = OUTPUT_CSV.with_name(OUTPUT_CSV.stem + '_summary.csv')
summary.to_csv(summary_path, index=False)
print(f'Wrote {summary_path}')
summary.head()